<a href="https://colab.research.google.com/github/VeenusDadri/training/blob/master/advance_pyspark/UDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Spark SQL').getOrCreate()
sc = spark.sparkContext

In [48]:
data=[(1,"maheer", 2000,500),(2,"mahi",4000,1000),(3,'himanshu', 5000, None )]
schema=['id','name', 'salary','bonus']
df=spark.createDataFrame(data,schema)
df.show()

+---+--------+------+-----+
| id|    name|salary|bonus|
+---+--------+------+-----+
|  1|  maheer|  2000|  500|
|  2|    mahi|  4000| 1000|
|  3|himanshu|  5000| NULL|
+---+--------+------+-----+



In [49]:
def total_pay(s,b):
  return s+b if s is not None and b is not None else 0
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
Total_Payment= udf(lambda x,y : total_pay(x,y), IntegerType())


In [50]:
df.withColumn('Total_Pay',Total_Payment(df.salary,df.bonus)).show()

+---+--------+------+-----+---------+
| id|    name|salary|bonus|Total_Pay|
+---+--------+------+-----+---------+
|  1|  maheer|  2000|  500|     2500|
|  2|    mahi|  4000| 1000|     5000|
|  3|himanshu|  5000| NULL|        0|
+---+--------+------+-----+---------+



In [51]:
T_P=udf(total_pay,IntegerType())
df.select(T_P(df.salary,df.bonus)).show()

+------------------------+
|total_pay(salary, bonus)|
+------------------------+
|                    2500|
|                    5000|
|                       0|
+------------------------+



In [52]:

@udf(returnType=IntegerType())
def total_pay(s,b):
  return s+b if s is not None and b is not None else 0



In [53]:
df.select('*', total_pay(df.salary,df.bonus)).show()

+---+--------+------+-----+------------------------+
| id|    name|salary|bonus|total_pay(salary, bonus)|
+---+--------+------+-----+------------------------+
|  1|  maheer|  2000|  500|                    2500|
|  2|    mahi|  4000| 1000|                    5000|
|  3|himanshu|  5000| NULL|                       0|
+---+--------+------+-----+------------------------+



In [54]:
#Below is the physical plan
df.select('*', total_pay(df.salary,df.bonus)).explain()

== Physical Plan ==
*(2) Project [id#613L, name#614, salary#615L, bonus#616L, pythonUDF0#712 AS total_pay(salary, bonus)#706]
+- BatchEvalPython [total_pay(salary#615L, bonus#616L)#705], [pythonUDF0#712]
   +- *(1) Scan ExistingRDD[id#613L,name#614,salary#615L,bonus#616L]




In [55]:
df.select('*', total_pay(df.salary,df.bonus).alias("Tot_Pay")).show()

+---+--------+------+-----+-------+
| id|    name|salary|bonus|Tot_Pay|
+---+--------+------+-----+-------+
|  1|  maheer|  2000|  500|   2500|
|  2|    mahi|  4000| 1000|   5000|
|  3|himanshu|  5000| NULL|      0|
+---+--------+------+-----+-------+



In [56]:
df.createOrReplaceTempView("emps")

In [57]:
df_result = spark.sql("SELECT * FROM emps")
df_result.show()

+---+--------+------+-----+
| id|    name|salary|bonus|
+---+--------+------+-----+
|  1|  maheer|  2000|  500|
|  2|    mahi|  4000| 1000|
|  3|himanshu|  5000| NULL|
+---+--------+------+-----+



In order to use the above function in spark sql, you need to register with spark.udf.register()

In [58]:
def total_pay(s,b):
  return s+b if s is not None and b is not None else 0
spark.udf.register(name='Total_Pay_SQL', f = total_pay,returnType=IntegerType() )

<function __main__.total_pay(s, b)>

In [59]:
df_result = spark.sql("SELECT id, name, Total_Pay_SQL(salary, bonus) FROM emps")
df_result.show()

+---+--------+----------------------------+
| id|    name|Total_Pay_SQL(salary, bonus)|
+---+--------+----------------------------+
|  1|  maheer|                        2500|
|  2|    mahi|                        5000|
|  3|himanshu|                           0|
+---+--------+----------------------------+



In [60]:
spark.udf.register("T_P_SQL",total_pay)

<function __main__.total_pay(s, b)>

In [61]:
df_result1 = spark.sql("SELECT id, name, T_P_SQL(salary, bonus) FROM emps")
df_result1.show()

+---+--------+----------------------+
| id|    name|T_P_SQL(salary, bonus)|
+---+--------+----------------------+
|  1|  maheer|                  2500|
|  2|    mahi|                  5000|
|  3|himanshu|                     0|
+---+--------+----------------------+



In [62]:
df_result = spark.sql("SELECT id, name, Total_Pay_SQL(salary, bonus) as TP FROM emps")
df_result.show()

+---+--------+----+
| id|    name|  TP|
+---+--------+----+
|  1|  maheer|2500|
|  2|    mahi|5000|
|  3|himanshu|   0|
+---+--------+----+



In [63]:
spark.udf.register("spark_square", lambda x: x**2, returnType=IntegerType())

spark.sql("select id, spark_square(id) from emps").show()

+---+----------------+
| id|spark_square(id)|
+---+----------------+
|  1|               1|
|  2|               4|
|  3|               9|
+---+----------------+



# UDFs with Complex DataTypes

In [64]:
df = spark.createDataFrame([[1, [3, 4, 5, 6]], [2, [4, 2]], [3, [9, 10, 11, 15, 17]], [4, None], [5, [98]]], ['id', 'arr'])

In [65]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StructType, StructField, IntegerType, ArrayType, MapType


struct_schema = StructType([
    StructField('size', IntegerType(), nullable=True),
    StructField('elements', ArrayType(IntegerType()), nullable=True)
])

@udf(returnType=struct_schema)
def calc_len(arr):
    if arr:
        return {'size': len(arr), 'elements': arr}

df.select('id', 'arr', calc_len('arr')).show(5, truncate=False)

+---+-------------------+------------------------+
|id |arr                |calc_len(arr)           |
+---+-------------------+------------------------+
|1  |[3, 4, 5, 6]       |{4, [3, 4, 5, 6]}       |
|2  |[4, 2]             |{2, [4, 2]}             |
|3  |[9, 10, 11, 15, 17]|{5, [9, 10, 11, 15, 17]}|
|4  |NULL               |NULL                    |
|5  |[98]               |{1, [98]}               |
+---+-------------------+------------------------+



In [66]:
df.select('id', 'arr', calc_len('arr')).printSchema()

root
 |-- id: long (nullable = true)
 |-- arr: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- calc_len(arr): struct (nullable = true)
 |    |-- size: integer (nullable = true)
 |    |-- elements: array (nullable = true)
 |    |    |-- element: integer (containsNull = true)



# Pyspark UDF with ArrayType

In [67]:
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType, ArrayType

@udf(returnType=ArrayType(FloatType()))
def float_arr(arr):
    if arr:
        return [i/2 for i in arr]

df.withColumn('float_arr', float_arr('arr')).show(5, truncate=False)

+---+-------------------+-------------------------+
|id |arr                |float_arr                |
+---+-------------------+-------------------------+
|1  |[3, 4, 5, 6]       |[1.5, 2.0, 2.5, 3.0]     |
|2  |[4, 2]             |[2.0, 1.0]               |
|3  |[9, 10, 11, 15, 17]|[4.5, 5.0, 5.5, 7.5, 8.5]|
|4  |NULL               |NULL                     |
|5  |[98]               |[49.0]                   |
+---+-------------------+-------------------------+



In [68]:
df2 = spark.createDataFrame([("10",), ("abc",)], ["value"])
df2.selectExpr("CAST(value AS INT)").show()


+-----+
|value|
+-----+
|   10|
| NULL|
+-----+



In [70]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr
df5 = spark.createDataFrame([({"key": "value"},), ({"number": 10},)], ["data"])
df5.withColumn("data_as_string", expr("CAST(data AS STRING)")).show()


+--------------+--------------+
|          data|data_as_string|
+--------------+--------------+
|{key -> value}|{key -> value}|
|{number -> 10}|{number -> 10}|
+--------------+--------------+



In [71]:

df4 = spark.createDataFrame([("Zimmy",), ("Snoopi",), ("puffy",)], ["name"])
df4.orderBy(col("name").asc()).show()

+------+
|  name|
+------+
|Snoopi|
| Zimmy|
| puffy|
+------+



In [72]:
df3 = spark.readStream.format("rate").option("rowsPerSecond", 1).load()
df3.selectExpr("timestamp", "value", "value % 2 as is_even").writeStream.format("console").start()